# 07 - E7 test-time adaptation

E7 evaluates source, TENT, and EATA-lite on the E6 winner checkpoint (`e6_ema_kmeans_restart`). TTA is reset at the start of each clean/corruption/severity condition.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

repo root: /home/nakato/projects/Dememte


In [2]:
from dataclasses import asdict
import math

import numpy as np
import pandas as pd
import torch

from dememte.config import E6_SPECS, e6_config
from dememte.data import build_loaders, seed_everything
from dememte.evaluation import evaluate_dememte_suite, evaluate_dememte_tta_suite, signal_curve_rows
from dememte.io import ensure_dir, load_checkpoint, write_csv, write_json
from dememte.models import make_dememte_e6
from dememte.tta import EATALiteAdapter, TentAdapter, collect_tta_bn_params, configure_tta_model, make_tta_optimizer

BASE_VARIANT = 'e6_ema_kmeans_restart'
METHODS = ['source', 'tent_bn', 'eata_lite_d005', 'eata_lite_d04']
OUT = ensure_dir(ROOT / 'notebooks' / '07_e7_tta' / 'out')
E6_OUT = ROOT / 'notebooks' / '06_e6_zq_alignment' / 'out'
CKPT = E6_OUT / BASE_VARIANT / 'best.pt'

cfg = e6_config(BASE_VARIANT)
for candidate in [ROOT / 'experiments' / 'data', ROOT / 'data', Path(cfg.data_dir).expanduser()]:
    candidate = candidate.resolve()
    if (candidate / 'flowers-102').exists() or candidate.name == 'flowers-102':
        cfg.data_dir = str(candidate)
        break

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed_everything(cfg.seed)
print('device:', device)
print('checkpoint:', CKPT)
print('eata entropy margin:', 0.4 * math.log(cfg.num_classes))

device: cuda
checkpoint: /home/nakato/projects/Dememte/notebooks/06_e6_zq_alignment/out/e6_ema_kmeans_restart/best.pt
eata entropy margin: 1.8499891253137084


## Data

In [3]:
tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=cfg.val_ratio,
    split_seed=cfg.split_seed,
    protocol=cfg.benchmark_protocol,
)
print(meta)

{'protocol': 'historical_trainval_resplit', 'split_seed': 42, 'train_size': 1632, 'val_size': 408, 'test_size': 6149}


## Evaluation helpers

In [4]:
def write_markdown_table(rows, path):
    path = Path(path)
    ensure_dir(path.parent)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    df = pd.DataFrame(rows)
    path.write_text(df.to_markdown(index=False), encoding='utf-8')


def load_base_model():
    model = make_dememte_e6(cfg, device=device)
    load_checkpoint(model, CKPT, device=device, strict=True)
    return model


def make_adapter(method):
    model = configure_tta_model(load_base_model())
    params, names = collect_tta_bn_params(model)
    if not params:
        raise RuntimeError('No BatchNorm affine parameters found for TTA')
    opt = make_tta_optimizer(params, lr=2.5e-4, momentum=0.9)
    if method == 'tent_bn':
        return TentAdapter(model, opt, steps=1, episodic=False)
    if method == 'eata_lite_d005':
        return EATALiteAdapter(model, opt, num_classes=cfg.num_classes, steps=1, episodic=False, d_margin=0.05)
    if method == 'eata_lite_d04':
        return EATALiteAdapter(model, opt, num_classes=cfg.num_classes, steps=1, episodic=False, d_margin=0.4)
    raise ValueError(method)


def summarize_metrics(method, metrics):
    summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
    summary.update({
        'variant': method,
        'label': method,
        'base_variant': BASE_VARIANT,
        'base_checkpoint': str(CKPT),
        'protocol': meta['protocol'],
        'split_seed': meta['split_seed'],
        'quantizer_type': cfg.quantizer_type,
    })
    return summary

## Run E7

In [5]:
if not CKPT.exists():
    raise FileNotFoundError(f'Missing E6 winner checkpoint: {CKPT}')

all_summaries = []
all_curves = []

for method in METHODS:
    print(f'=== {method} ===')
    if method == 'source':
        model = load_base_model()
        metrics = evaluate_dememte_suite(model, te_loader, device=device)
    else:
        metrics = evaluate_dememte_tta_suite(
            lambda method=method: make_adapter(method),
            te_loader,
            device=device,
            tta_method=method,
            tta_base_variant=BASE_VARIANT,
        )

    clean_record = metrics.pop('clean_record')
    corrupt_records = metrics.pop('corruption_records')
    curve_rows = signal_curve_rows(method, method, clean_record, corrupt_records)
    summary = summarize_metrics(method, metrics)
    all_summaries.append(summary)
    all_curves.extend(curve_rows)

    method_dir = ensure_dir(OUT / method)
    write_json(summary, method_dir / 'metrics.json')
    write_csv(curve_rows, method_dir / 'signal_curves.csv')
    print({k: summary[k] for k in ['clean_acc', 'corrupt_acc_avg', 'ece_corrupt_avg', 'nll_corrupt_avg'] if k in summary})

write_csv(all_summaries, OUT / 'e7_results.csv')
write_csv(all_curves, OUT / 'e7_curves.csv')

ranked = sorted(all_summaries, key=lambda r: r.get('corrupt_acc_avg', 0.0), reverse=True)
write_markdown_table(ranked, OUT / 'e7_summary.md')
pd.DataFrame(ranked)

=== source ===
{'clean_acc': 0.7523174499918686, 'corrupt_acc_avg': 0.502954409931154, 'ece_corrupt_avg': 0.09026845803041757, 'nll_corrupt_avg': 2.022439048737882}
=== tent_bn ===
{'clean_acc': 0.0318751016425435, 'corrupt_acc_avg': 0.028608987911313492, 'ece_corrupt_avg': 0.5089828923917936, 'nll_corrupt_avg': 8.988141024012933}
=== eata_lite_d005 ===
{'clean_acc': 0.02618311920637502, 'corrupt_acc_avg': 0.021995446414051066, 'ece_corrupt_avg': 0.5127675231489014, 'nll_corrupt_avg': 9.22715085746995}
=== eata_lite_d04 ===
{'clean_acc': 0.031061961294519436, 'corrupt_acc_avg': 0.027606114815417138, 'ece_corrupt_avg': 0.5359362059429787, 'nll_corrupt_avg': 9.34639151309026}


,clean_acc,corrupt_acc_avg,corrupt_acc_gaussian_noise,corrupt_acc_pixel_mask,corrupt_acc_cutout,corrupt_acc_blur,ece_clean,ece_corrupt_avg,nll_clean,nll_corrupt_avg,...,base_checkpoint,protocol,split_seed,quantizer_type,tta_updates_clean,tta_updates_corrupt_avg,tta_selection_rate_clean,tta_selection_rate_corrupt_avg,tta_method,tta_base_variant
0,0.752317,0.502954,0.353445,0.348675,0.638911,0.670787,0.058221,0.090268,0.976969,2.022439,...,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,NaN,NaN,NaN,NaN,NaN,NaN
1,0.031875,0.028609,0.026888,0.023039,0.031767,0.032742,0.508754,0.508983,8.866587,8.988141,...,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,385.0,385.000000,1.000000,1.000000,tent_bn,e6_ema_kmeans_restart
2,0.031062,0.027606,0.024774,0.022443,0.032417,0.030791,0.527907,0.535936,9.120397,9.346392,...,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,385.0,385.000000,0.629371,0.616469,eata_lite_d04,e6_ema_kmeans_restart
3,0.026183,0.021995,0.019841,0.018756,0.024557,0.024828,0.514376,0.512768,9.049344,9.227151,...,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,311.0,316.416667,0.108798,0.109909,eata_lite_d005,e6_ema_kmeans_restart
